# Notebook 03_2 — Predictive Performance (Text-Only Embeddings)

**Replication of Bach et al. (2025) — Adventures in Demand Analysis Using AI**  
Applied to: Amazon Men's Shoes (Size 8)

---

This notebook evaluates how well different model specifications predict:
- **Q_t** — Sales rank (demand proxy)
- **P_bb_t** — Buybox price

Model specifications compared (following paper Table 2):
- OLS / Boosting with Tabular features only
- OLS / Boosting with Tabular + PCA (5 components)
- OLS / Boosting with Tabular + Cluster Similarities
- Deep Time Independent (RoBERTa text embeddings)
- Deep Time Dependent (RoBERTa text + lag1 embeddings)

R² evaluated on both level (Q_t, P_bb_t) and first-difference (ΔQ_t, ΔP_bb_t) models.

## ① Mount Drive

In [1]:
# Drive mount not needed for local execution
print('✅ Local mode')

✅ Local mode


## ② Set Working Directory

In [2]:
import os, sys
from pathlib import Path

# Detect project root by walking up from CWD to find 'data/' and 'code/' folders
_cwd = Path.cwd()
PROJECT_ROOT = _cwd
for _ in range(5):
    if (PROJECT_ROOT / 'data').is_dir() and (PROJECT_ROOT / 'code').is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Cannot find project root (expected 'data/' and 'code/' folders)")

# Change to code/ directory so relative imports and paths work
CODE_DIR = str(PROJECT_ROOT / 'code')
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)

print(f'Working directory: {os.getcwd()}')
print(f'Files here: {os.listdir(".")}')

Working directory: /home/iankuzuma/claude_code/demand_modeling/men-8-subcat-split-separate-embedding/oxfords/code
Files here: ['utils', 'main_train_keys.csv', 'main_val_keys.csv', 'requirements.txt', '01_1_create_dataset_txt_img.ipynb', '01_2_create_dataset_txt.ipynb', '02_cluster_centroid_products.ipynb', '02_cluster_centroid_products_random20.ipynb', '03_1_predictive_performance_txt_img.ipynb', '03_2_predictive_performance_txt.ipynb', '04_evaluation.ipynb']


## ③ Imports

In [3]:
import re
import datasets
import pandas as pd
import numpy as np
import statsmodels.api as sm

from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

from utils.utils_data2 import (
    load_pred_and_emb,
    center_and_norm,
    get_pca,
    get_cluster,
    get_similarities,
    add_lags_and_scale_data,
)
print('✅ Imports done')

✅ Imports done


## ④ Load Dataset

Loads the text-only embeddings dataset from notebook 01_2.
`dropna()` removes first-period rows with no lag features.

In [4]:
txt_only = True
embeddings = True
dataframe_name = f"dataset_txt_only_{txt_only}_embeddings_{embeddings}"

df_full_train = pd.read_csv(f"../data/{dataframe_name}_train.zip")
df_full_val   = pd.read_csv(f"../data/{dataframe_name}_val.zip")

columns_to_drop = [
    "Q_t-2", "P_bb_t-2", "REVIEW_COUNT_t-2", "RATING_t-2",
]
df_full_train = df_full_train.drop(columns=columns_to_drop)
df_full_val   = df_full_val.drop(columns=columns_to_drop)

df_full_train = df_full_train.dropna()
df_full_val   = df_full_val.dropna()

dummy_subcat_names = [category for category in df_full_val["subcat_aggregated"].unique()]
all_time_steps = sorted([str(date) for date in df_full_val["date"].unique()])

print(f"Train shape: {df_full_train.shape}")
print(f"Val shape:   {df_full_val.shape}")
print(f"Dummy subcat names: {dummy_subcat_names}")
print(f"All time steps: {all_time_steps}")
df_full_val.columns

Train shape: (1560, 322)
Val shape:   (1572, 322)
Dummy subcat names: ['Oxfords']
All time steps: ['2025-04-28', '2025-05-26', '2025-06-23', '2025-07-21', '2025-08-18', '2025-09-15', '2025-10-13', '2025-11-10', '2025-12-08', '2026-01-05', '2026-02-02', '2026-03-02']


Index(['ASIN', 'date', 'Q_t', 'PRICE', 'P_bb_t', 'text', 'window',
       'REVIEW_COUNT', 'RATING', 'New Offer Count: Current',
       ...
       'Delta_Q_t', 'Delta_P_bb_t', 'pred_ml_l', 'pred_ml_m', 'pred_ml_l_diff',
       'pred_ml_m_diff', 'pred_ml_l_lag_1', 'pred_ml_m_lag_1',
       'pred_ml_l_diff_lag_1', 'pred_ml_m_diff_lag_1'],
      dtype='object', length=322)

## ⑤ Define Controls and Feature Specifications

Controls follow paper Section 3:
- Continuous: RATING, REVIEW_COUNT, offer counts
- Time dummies (excluding first period as baseline)
- Subcat dummies (excluding Residual as baseline)
- PCA components (5) and cluster similarities (5) as additional features

In [5]:
n_lags = 1

outcome   = "Q_t"
treatment = "P_bb_t"

outcome_diff   = "Delta_Q_t"
treatment_diff = "Delta_P_bb_t"

dummy_time_steps = all_time_steps[n_lags:]

all_dummy_controls = (
    dummy_subcat_names
    + dummy_time_steps
    + ["Lightning Deals: Upcoming Deal", "Buy Box: Is FBA"]
)

dummy_baselines = [dummy_time_steps[0], "Residual"]
dummy_time_steps_wo_baseline = [t for t in dummy_time_steps if t not in dummy_baselines]
dummy_controls = [t for t in all_dummy_controls if t not in dummy_baselines]

cont_controls = [
    "RATING_t-1",
    "REVIEW_COUNT_t-1",
    "New Offer Count: Current",
    "Count of retrieved live offers: New, FBA",
    "Count of retrieved live offers: New, FBM",
]

add_controls_to_scale = ["RATING", "REVIEW_COUNT"]

controls_pca = ["pca_0", "pca_1", "pca_2", "pca_3", "pca_4"]
controls_similarities = [
    "similarity_cluster_0", "similarity_cluster_1", "similarity_cluster_2",
    "similarity_cluster_3", "similarity_cluster_4",
]

additional_controls = cont_controls + dummy_controls
additional_controls_deep = [var for var in additional_controls if var not in dummy_subcat_names]
controls_emb = [var for var in df_full_train.columns if "emb" in var]

print(f"Continuous controls:    {len(cont_controls)}")
print(f"Dummy controls:         {len(dummy_controls)}")
print(f"Total controls:         {len(additional_controls)}")
print(f"Embedding columns:      {len(controls_emb)}")

Continuous controls:    5
Dummy controls:         13
Total controls:         18
Embedding columns:      256


## ⑥ Initialize Results DataFrames

In [6]:
column_names = ["R2 Q Train", "R2 Q Test", "R2 P Train", "R2 P Test"]

results_df      = pd.DataFrame(columns=column_names)
results_df_diff = pd.DataFrame(columns=column_names)
print('✅ Results DataFrames initialized')

✅ Results DataFrames initialized


## ⑦ Deep Model R² — Level

Evaluates how well the deep learning predictions (from Part 5) explain
sales rank and price in both time-independent and lag1 configurations.

In [7]:
df_dict = {"Train": df_full_train, "Test": df_full_val}

results_df_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y = df[outcome].values
    d = df[treatment].values

    pred_ml_l = df["pred_ml_l"].values
    pred_ml_m = df["pred_ml_m"].values

    r2_ml_l = np.round(r2_score(y, pred_ml_l), 4)
    r2_ml_m = np.round(r2_score(d, pred_ml_m), 4)

    print(f"Evaluation for {df_name} set")
    print(f"  R2 Outcome (time independent):  {r2_ml_l}")
    print(f"  R2 Treatment (time independent): {r2_ml_m}")

    pred_ml_l_lag1 = df["pred_ml_l_lag_1"].values
    pred_ml_m_lag1 = df["pred_ml_m_lag_1"].values

    r2_ml_l_lag1 = np.round(r2_score(y, pred_ml_l_lag1), 4)
    r2_ml_m_lag1 = np.round(r2_score(d, pred_ml_m_lag1), 4)

    print(f"  R2 Outcome (lag1):               {r2_ml_l_lag1}")
    print(f"  R2 Treatment (lag1):             {r2_ml_m_lag1}")
    print()

    results_df_deep[f"R2 Q {df_name}"] = (r2_ml_l, r2_ml_l_lag1)
    results_df_deep[f"R2 P {df_name}"] = (r2_ml_m, r2_ml_m_lag1)

results_df_deep

Evaluation for Train set
  R2 Outcome (time independent):  0.4605
  R2 Treatment (time independent): -0.2783
  R2 Outcome (lag1):               0.6881
  R2 Treatment (lag1):             -0.1435

Evaluation for Test set
  R2 Outcome (time independent):  0.4015
  R2 Treatment (time independent): -0.2614
  R2 Outcome (lag1):               0.6692
  R2 Treatment (lag1):             -0.1214



,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
Deep Time Independent,0.4605,0.4015,-0.2783,-0.2614
Deep Time Dependent,0.6881,0.6692,-0.1435,-0.1214


## ⑧ Deep Model R² — Diff

Same evaluation on first-difference outcomes (ΔQ_t, ΔP_bb_t).

In [8]:
results_df_diff_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y_diff = df["Delta_Q_t"].values
    d_diff = df["Delta_P_bb_t"].values

    pred_ml_l_diff = df["pred_ml_l_diff"].values
    pred_ml_m_diff = df["pred_ml_m_diff"].values

    r2_ml_l_diff = np.round(r2_score(y_diff, pred_ml_l_diff), 8)
    r2_ml_m_diff = np.round(r2_score(d_diff, pred_ml_m_diff), 8)

    print(f"Evaluation for {df_name} set")
    print(f"  R2 Outcome diff (time independent):  {r2_ml_l_diff}")
    print(f"  R2 Treatment diff (time independent): {r2_ml_m_diff}")

    pred_ml_l_diff_lag1 = df["pred_ml_l_diff_lag_1"].values
    pred_ml_m_diff_lag1 = df["pred_ml_m_diff_lag_1"].values

    r2_ml_l_diff_lag1 = np.round(r2_score(y_diff, pred_ml_l_diff_lag1), 8)
    r2_ml_m_diff_lag1 = np.round(r2_score(d_diff, pred_ml_m_diff_lag1), 8)

    print(f"  R2 Outcome diff (lag1):               {r2_ml_l_diff_lag1}")
    print(f"  R2 Treatment diff (lag1):             {r2_ml_m_diff_lag1}")
    print()

    results_df_diff_deep[f"R2 Q {df_name}"] = (r2_ml_l_diff, r2_ml_l_diff_lag1)
    results_df_diff_deep[f"R2 P {df_name}"] = (r2_ml_m_diff, r2_ml_m_diff_lag1)

results_df_diff_deep

Evaluation for Train set
  R2 Outcome diff (time independent):  0.0025165
  R2 Treatment diff (time independent): -0.00526211
  R2 Outcome diff (lag1):               0.01394317
  R2 Treatment diff (lag1):             -0.0035737

Evaluation for Test set
  R2 Outcome diff (time independent):  0.0008443
  R2 Treatment diff (time independent): -0.00531117
  R2 Outcome diff (lag1):               0.01130028
  R2 Treatment diff (lag1):             -0.00395566



,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
Deep Time Independent,0.002517,0.000844,-0.005262,-0.005311
Deep Time Dependent,0.013943,0.011300,-0.003574,-0.003956


## ⑨ Build Feature Matrices

Constructs train and test feature matrices for tabular, PCA, and similarity
specifications used in OLS and Boosting models.

In [9]:
# Train set
y_train       = df_full_train[outcome].squeeze()
d_train       = df_full_train[treatment].squeeze()
y_train_diff  = df_full_train[outcome_diff].squeeze()
d_train_diff  = df_full_train[treatment_diff].squeeze()

x_train     = sm.add_constant(df_full_train[additional_controls])
x_train_pca = sm.add_constant(df_full_train[additional_controls + controls_pca])
x_train_sim = sm.add_constant(df_full_train[additional_controls + controls_similarities])

# Test set
y_test      = df_full_val[outcome].squeeze()
d_test      = df_full_val[treatment].squeeze()
y_test_diff = df_full_val[outcome_diff].squeeze()
d_test_diff = df_full_val[treatment_diff].squeeze()

x_test     = sm.add_constant(df_full_val[additional_controls])
x_test_pca = sm.add_constant(df_full_val[additional_controls + controls_pca])
x_test_sim = sm.add_constant(df_full_val[additional_controls + controls_similarities])

# Rename columns for LightGBM (no special characters)
for df in [x_train, x_test, x_train_pca, x_test_pca, x_train_sim, x_test_sim]:
    df.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '', x), inplace=True)

print(f"x_train shape:     {x_train.shape}")
print(f"x_train_pca shape: {x_train_pca.shape}")
print(f"x_train_sim shape: {x_train_sim.shape}")

x_train shape:     (1560, 18)
x_train_pca shape: (1560, 23)
x_train_sim shape: (1560, 23)


## ⑩ Build Dict Structures

In [10]:
train_dict = {
    "y": y_train, "y_diff": y_train_diff,
    "d": d_train, "d_diff": d_train_diff,
    "x": x_train, "x_pca": x_train_pca, "x_sim": x_train_sim,
}

test_dict = {
    "y": y_test, "y_diff": y_test_diff,
    "d": d_test, "d_diff": d_test_diff,
    "x": x_test, "x_pca": x_test_pca, "x_sim": x_test_sim,
}
print('✅ Train and test dicts ready')

✅ Train and test dicts ready


## ⑪ Tabular Models — Level

Compares OLS and LightGBM Boosting across three feature specifications:
- **Tabular** — continuous + time/subcat dummies only
- **Tabular + PCA** — adds 5 PCA components from embeddings
- **Tabular + Similarities** — adds 5 cluster similarity scores

In [11]:
feature_specifications = ["x", "x_pca", "x_sim"]
results_df_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print(f"\n Feature specification: {feature_specification}")
    print("  OLS")
    outcome_reg = sm.OLS(train_dict["y"], train_dict[feature_specification]).fit()
    ols_outcome_train   = r2_score(train_dict["y"], outcome_reg.predict(train_dict[feature_specification]))
    ols_outcome_test    = r2_score(test_dict["y"],  outcome_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Outcome train/test: {ols_outcome_train:.4f} / {ols_outcome_test:.4f}")

    treatment_reg = sm.OLS(train_dict["d"], train_dict[feature_specification]).fit()
    ols_treatment_train = r2_score(train_dict["d"], treatment_reg.predict(train_dict[feature_specification]))
    ols_treatment_test  = r2_score(test_dict["d"],  treatment_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Treatment train/test: {ols_treatment_train:.4f} / {ols_treatment_test:.4f}")

    print("  Boosting")
    boost_q = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_q.fit(train_dict[feature_specification], train_dict["y"])
    r2_boost_q_train = boost_q.score(train_dict[feature_specification], train_dict["y"])
    r2_boost_q_test  = boost_q.score(test_dict[feature_specification],  test_dict["y"])
    print(f"  R2 Outcome train/test: {r2_boost_q_train:.4f} / {r2_boost_q_test:.4f}")

    boost_d = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_d.fit(train_dict[feature_specification], train_dict["d"])
    r2_boost_d_train = boost_d.score(train_dict[feature_specification], train_dict["d"])
    r2_boost_d_test  = boost_d.score(test_dict[feature_specification],  test_dict["d"])
    print(f"  R2 Treatment train/test: {r2_boost_d_train:.4f} / {r2_boost_d_test:.4f}")

    results_df_tab = pd.concat([results_df_tab, pd.DataFrame(data=[
        [ols_outcome_train, ols_outcome_test, ols_treatment_train, ols_treatment_test],
        [r2_boost_q_train,  r2_boost_q_test,  r2_boost_d_train,   r2_boost_d_test],
    ])], axis=0)

row_names = [
    "OLS (Tabular)", "Boosting (Tabular)",
    "OLS (Tabular + PCA)", "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)", "Boosting (Tabular + Similarities)",
]
results_df_tab.columns = column_names
results_df_tab.index   = row_names
results_df_tab


 Feature specification: x
  OLS
  R2 Outcome train/test: 0.3504 / 0.2893
  R2 Treatment train/test: 0.0620 / 0.0459
  Boosting


  R2 Outcome train/test: 0.8845 / 0.6075


  R2 Treatment train/test: 0.5811 / -0.0574

 Feature specification: x_pca
  OLS
  R2 Outcome train/test: 0.5398 / 0.4351
  R2 Treatment train/test: 0.1271 / 0.1302
  Boosting


  R2 Outcome train/test: 0.9676 / 0.5727


  R2 Treatment train/test: 0.8762 / 0.0719

 Feature specification: x_sim
  OLS
  R2 Outcome train/test: 0.5403 / 0.4739
  R2 Treatment train/test: 0.1267 / 0.1465
  Boosting


  R2 Outcome train/test: 0.9638 / 0.5358


  R2 Treatment train/test: 0.8293 / -0.0853


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.350382,0.289319,0.061991,0.045937
Boosting (Tabular),0.884503,0.607508,0.581078,-0.057411
OLS (Tabular + PCA),0.539797,0.435136,0.127064,0.130199
Boosting (Tabular + PCA),0.967583,0.572695,0.876195,0.071924
OLS (Tabular + Similarities),0.540284,0.473922,0.126678,0.146466
Boosting (Tabular + Similarities),0.963830,0.535825,0.829317,-0.085333


## ⑫ Summary — Level Models

In [12]:
results_df = pd.concat([results_df_tab, results_df_deep], axis=0)
results_df

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.350382,0.289319,0.061991,0.045937
Boosting (Tabular),0.884503,0.607508,0.581078,-0.057411
OLS (Tabular + PCA),0.539797,0.435136,0.127064,0.130199
Boosting (Tabular + PCA),0.967583,0.572695,0.876195,0.071924
OLS (Tabular + Similarities),0.540284,0.473922,0.126678,0.146466
Boosting (Tabular + Similarities),0.963830,0.535825,0.829317,-0.085333
Deep Time Independent,0.460500,0.401500,-0.278300,-0.261400
Deep Time Dependent,0.688100,0.669200,-0.143500,-0.121400


## ⑬ Tabular Models — Diff

Same comparison on first-difference outcomes (ΔQ_t, ΔP_bb_t).
First differences remove product fixed effects and time trends.

In [13]:
feature_specifications = ["x", "x_pca", "x_sim"]
results_df_diff_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print(f"\n Feature specification: {feature_specification}")
    print("  OLS")
    outcome_reg = sm.OLS(train_dict["y_diff"], train_dict[feature_specification]).fit()
    ols_outcome_train   = r2_score(train_dict["y_diff"], outcome_reg.predict(train_dict[feature_specification]))
    ols_outcome_test    = r2_score(test_dict["y_diff"],  outcome_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Outcome diff train/test: {ols_outcome_train:.4f} / {ols_outcome_test:.4f}")

    treatment_reg = sm.OLS(train_dict["d_diff"], train_dict[feature_specification]).fit()
    ols_treatment_train = r2_score(train_dict["d_diff"], treatment_reg.predict(train_dict[feature_specification]))
    ols_treatment_test  = r2_score(test_dict["d_diff"],  treatment_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Treatment diff train/test: {ols_treatment_train:.4f} / {ols_treatment_test:.4f}")

    print("  Boosting")
    boost_q = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_q.fit(train_dict[feature_specification], train_dict["y_diff"])
    r2_boost_q_train = boost_q.score(train_dict[feature_specification], train_dict["y_diff"])
    r2_boost_q_test  = boost_q.score(test_dict[feature_specification],  test_dict["y_diff"])
    print(f"  R2 Outcome diff train/test: {r2_boost_q_train:.4f} / {r2_boost_q_test:.4f}")

    boost_d = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_d.fit(train_dict[feature_specification], train_dict["d_diff"])
    r2_boost_d_train = boost_d.score(train_dict[feature_specification], train_dict["d_diff"])
    r2_boost_d_test  = boost_d.score(test_dict[feature_specification],  test_dict["d_diff"])
    print(f"  R2 Treatment diff train/test: {r2_boost_d_train:.4f} / {r2_boost_d_test:.4f}")

    results_df_diff_tab = pd.concat([results_df_diff_tab, pd.DataFrame(data=[
        [ols_outcome_train, ols_outcome_test, ols_treatment_train, ols_treatment_test],
        [r2_boost_q_train,  r2_boost_q_test,  r2_boost_d_train,   r2_boost_d_test],
    ])], axis=0)

row_names = [
    "OLS (Tabular)", "Boosting (Tabular)",
    "OLS (Tabular + PCA)", "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)", "Boosting (Tabular + Similarities)",
]
results_df_diff_tab.columns = column_names
results_df_diff_tab.index   = row_names
results_df_diff_tab


 Feature specification: x
  OLS
  R2 Outcome diff train/test: 0.1543 / 0.2115
  R2 Treatment diff train/test: 0.0161 / -0.0053
  Boosting


  R2 Outcome diff train/test: 0.5654 / 0.3552


  R2 Treatment diff train/test: 0.3700 / -0.1569

 Feature specification: x_pca
  OLS
  R2 Outcome diff train/test: 0.1568 / 0.2058
  R2 Treatment diff train/test: 0.0165 / -0.0067
  Boosting


  R2 Outcome diff train/test: 0.7031 / 0.3471


  R2 Treatment diff train/test: 0.4782 / -0.1044

 Feature specification: x_sim
  OLS
  R2 Outcome diff train/test: 0.1568 / 0.2044
  R2 Treatment diff train/test: 0.0170 / -0.0060
  Boosting


  R2 Outcome diff train/test: 0.6662 / 0.3119


  R2 Treatment diff train/test: 0.4154 / -0.1161


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.154297,0.211513,0.016121,-0.005288
Boosting (Tabular),0.565421,0.355210,0.370041,-0.156897
OLS (Tabular + PCA),0.156808,0.205850,0.016501,-0.006737
Boosting (Tabular + PCA),0.703069,0.347149,0.478230,-0.104429
OLS (Tabular + Similarities),0.156840,0.204362,0.017012,-0.006026
Boosting (Tabular + Similarities),0.666199,0.311870,0.415426,-0.116062


## ⑭ Summary — Diff Models

In [14]:
results_df_diff = pd.concat([results_df_diff_tab, results_df_diff_deep], axis=0)
results_df_diff

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.154297,0.211513,0.016121,-0.005288
Boosting (Tabular),0.565421,0.355210,0.370041,-0.156897
OLS (Tabular + PCA),0.156808,0.205850,0.016501,-0.006737
Boosting (Tabular + PCA),0.703069,0.347149,0.478230,-0.104429
OLS (Tabular + Similarities),0.156840,0.204362,0.017012,-0.006026
Boosting (Tabular + Similarities),0.666199,0.311870,0.415426,-0.116062
Deep Time Independent,0.002517,0.000844,-0.005262,-0.005311
Deep Time Dependent,0.013943,0.011300,-0.003574,-0.003956


## ⑮ Final Summary (% format)

R² values multiplied by 100 for readability — matches paper Table 2 format.

In [15]:
print("=== Level Models — Test R² (%) ===")
print(results_df[["R2 Q Test", "R2 P Test"]].round(4) * 100)
print()
print("=== Diff Models — Test R² (%) ===")
print(results_df_diff[["R2 Q Test", "R2 P Test"]].round(4) * 100)

=== Level Models — Test R² (%) ===
                                   R2 Q Test  R2 P Test
OLS (Tabular)                          28.93       4.59
Boosting (Tabular)                     60.75      -5.74
OLS (Tabular + PCA)                    43.51      13.02
Boosting (Tabular + PCA)               57.27       7.19
OLS (Tabular + Similarities)           47.39      14.65
Boosting (Tabular + Similarities)      53.58      -8.53
Deep Time Independent                  40.15     -26.14
Deep Time Dependent                    66.92     -12.14

=== Diff Models — Test R² (%) ===
                                   R2 Q Test  R2 P Test
OLS (Tabular)                          21.15      -0.53
Boosting (Tabular)                     35.52     -15.69
OLS (Tabular + PCA)                    20.58      -0.67
Boosting (Tabular + PCA)               34.71     -10.44
OLS (Tabular + Similarities)           20.44      -0.60
Boosting (Tabular + Similarities)      31.19     -11.61
Deep Time Independent             